<a href="https://colab.research.google.com/github/irfanalam05/Machine-Learning/blob/main/Fraud_Detector_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv("fraud_dataset.csv")

df.head()

,text,label
0,Your bank account will be blocked immediately,1
1,Share your OTP to avoid suspension,1
2,RBI verification required urgently,1
3,Your KYC has expired update now,1
4,Click this refund link immediately,1


In [ ]:
df.shape

(100, 2)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df["text"])
y = df["label"]

print("Text converted to numbers successfully!")

Text converted to numbers successfully!


In [ ]:
from sklearn.linear_model import LogisticRegression

# Create model
model = LogisticRegression()

# Train model using X (text numbers) and y (labels)
model.fit(X, y)

print("Model trained successfully!")


Model trained successfully!


In [ ]:
from sklearn.metrics import accuracy_score

predictions = model.predict(X)
accuracy = accuracy_score(y, predictions)

print("Training Accuracy:", accuracy)


Training Accuracy: 1.0


In [ ]:
#Test the Model
def check_fraud(text):
    X_test = vectorizer.transform([text])
    probability = model.predict_proba(X_test)[0][1]
    return probability

print("Fraud Score:", check_fraud("Share your OTP immediately"))
print("Fraud Score:", check_fraud("Let's meet tomorrow"))

Fraud Score: 0.6732410790888161
Fraud Score: 0.27985310173150374


In [ ]:
from sklearn.metrics import accuracy_score

predictions = model.predict(X)
accuracy = accuracy_score(y, predictions)

print("Training Accuracy:", accuracy)

Training Accuracy: 1.0


Add Smart Fraud Checker Function

In [ ]:
fraud_keywords = ["otp", "kyc", "blocked", "urgent", "verify", "refund", "account", "suspended"]

def analyze_call(text):
    X_test = vectorizer.transform([text])
    probability = model.predict_proba(X_test)[0][1]

    detected_keywords = []
    for word in fraud_keywords:
        if word in text.lower():
            detected_keywords.append(word)

    if probability > 0.5:
        result = "FRAUD ALERT"
    else:
        result = "SAFE MESSAGE"

    return {
        "Fraud Probability": round(probability, 2),
        "Result": result,
        "Detected Risk Words": detected_keywords
    }


In [ ]:
print(analyze_call("Share your OTP immediately your account will be blocked"))
print(analyze_call("Let's meet tomorrow for lunch"))

{'Fraud Probability': np.float64(0.74), 'Result': 'FRAUD ALERT', 'Detected Risk Words': ['otp', 'blocked', 'account']}
{'Fraud Probability': np.float64(0.29), 'Result': 'SAFE MESSAGE', 'Detected Risk Words': []}


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test_split, y_train, y_test_split = train_test_split(X, y, test_size=0.2, random_state=42)

model2 = LogisticRegression()
model2.fit(X_train, y_train)

predictions = model2.predict(X_test_split)

from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test_split, predictions)

print("Test Accuracy:", accuracy)


Test Accuracy: 0.95


Add Audio → Text → Fraud Detection

In [ ]:
!pip install openai-whisper
!pip install torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.8 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=7df4600a0d969bcf7996992773612de621e89012901d93da5b8d4e2e36276e29
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
import whisper

# Load whisper model
model_whisper = whisper.load_model("base")

# Transcribe audio
result = model_whisper.transcribe("audio1.mp3")

# Print transcript
print("Transcribed Text:")
print(result["text"])


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 145MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcribed Text:
 Hello, very good afternoon. So, this I calling from bank, I mean this is your bank account has been free, so for that you need to be confirmation and for the same your attribute over does a suspension and I have a verification used required for this. You have to update your panic or immediately.


Pass Audio Text to Fraud Detector

In [ ]:
# Get transcript text
transcribed_text = result["text"]

# Analyze using your fraud model
analysis = analyze_call(transcribed_text)

print("----- FRAUD ANALYSIS RESULT -----")
print("Transcribed Text:", transcribed_text)
print("Fraud Probability:", analysis["Fraud Probability"])
print("Result:", analysis["Result"])
print("Detected Risk Words:", analysis["Detected Risk Words"])

----- FRAUD ANALYSIS RESULT -----
Transcribed Text:  Hello, very good afternoon. So, this I calling from bank, I mean this is your bank account has been free, so for that you need to be confirmation and for the same your attribute over does a suspension and I have a verification used required for this. You have to update your panic or immediately.
Fraud Probability: 0.56
Result: FRAUD ALERT
Detected Risk Words: ['account']
